# 04. LOTUSDIS ASR Baseline Check

## Goal

Evaluate `openai/whisper-small` on LOTUSDIS and keep the baseline as the usable local ASR result.

## Why This Step Matters

Local debug fine-tuning collapsed the model output into punctuation-like text, so this notebook records the working baseline and stops before harmful training.


## Inputs

- Streamed `nectec/LOTUSDIS` validation samples from Hugging Face.
- `openai/whisper-small` as the baseline ASR model.
- Local ASR normalization and metric helpers.

## Outputs

- Baseline CER and WER on a small validation subset.
- Baseline predictions for inspection.


## Setup

Fine-tuning is intentionally not included in this notebook. If fine-tuning rows appear, the kernel is stale or the notebook was overwritten by an old browser state.


In [8]:
from pathlib import Path
import re
import sys

import pandas as pd
import torch
from datasets import load_dataset
from transformers import WhisperForConditionalGeneration, WhisperProcessor

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.asr.metrics import character_error_rate, word_error_rate
from src.asr.normalize_text import normalize_text

MODEL_NAME = 'openai/whisper-small'
MAX_DEV_SAMPLES = 20
GENERATION_KWARGS = {
    'language': 'thai',
    'task': 'transcribe',
    'max_new_tokens': 80,
    'no_repeat_ngram_size': 3,
    'repetition_penalty': 1.05,
}
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32

print('Project root:', PROJECT_ROOT)
print('Device:', DEVICE)
print('Model dtype:', MODEL_DTYPE)
print('Fine-tuning: disabled')


Project root: /home/lataeq/Code/Saranote
Device: cuda
Model dtype: torch.float16
Fine-tuning: disabled


## Load Data


In [9]:
def decode_audio(audio):
    if hasattr(audio, 'get_all_samples'):
        samples = audio.get_all_samples()
        return samples.data.squeeze().detach().cpu().numpy(), samples.sample_rate

    return audio['array'], audio['sampling_rate']


def collect_samples(split: str, count: int) -> list[dict[str, object]]:
    rows = []
    dataset = load_dataset('nectec/LOTUSDIS', split=split, streaming=True)

    for sample in dataset:
        audio_array, sampling_rate = decode_audio(sample['audio'])
        rows.append(
            {
                'audio': audio_array,
                'sampling_rate': sampling_rate,
                'sentence': sample['sentence'],
            }
        )

        if len(rows) == count:
            break

    return rows


dev_samples = collect_samples('validation', MAX_DEV_SAMPLES)

print(f'Loaded {len(dev_samples)} validation samples')
print('Example reference:', dev_samples[0]['sentence'])


Loaded 20 validation samples
Example reference: ครับสวัสดีครับผม M สามสิบแปดครับ


## Run Experiment


In [10]:
THAI_DIGIT_WORDS = [
    'ศูนย์',
    'หนึ่ง',
    'สอง',
    'สาม',
    'สี่',
    'ห้า',
    'หก',
    'เจ็ด',
    'แปด',
    'เก้า',
]


def thai_number_under_100(value: int) -> str:
    if value < 10:
        return THAI_DIGIT_WORDS[value]
    if value < 20:
        return 'สิบ' + ('เอ็ด' if value == 11 else THAI_DIGIT_WORDS[value % 10] if value % 10 else '')

    tens_words = ['', '', 'ยี่สิบ', 'สามสิบ', 'สี่สิบ', 'ห้าสิบ', 'หกสิบ', 'เจ็ดสิบ', 'แปดสิบ', 'เก้าสิบ']
    unit = value % 10
    return tens_words[value // 10] + ('เอ็ด' if unit == 1 else THAI_DIGIT_WORDS[unit] if unit else '')


def normalize_number_token(match: re.Match[str]) -> str:
    token = match.group()
    if len(token) > 1 and token.startswith('0'):
        return ''.join(THAI_DIGIT_WORDS[int(digit)] for digit in token)

    value = int(token)
    return thai_number_under_100(value) if value < 100 else token


def prepare_target_text(text: str) -> str:
    replacements = {
        'M': 'เอ็ม',
        'F': 'เอฟ',
        'Topic': 'หัวข้อ',
        'Menu': 'เมนู',
        'Ok': 'โอเค',
    }

    for source, target in replacements.items():
        text = re.sub(rf'\b{source}\b', target, text, flags=re.IGNORECASE)

    text = re.sub(r'(?i)(?<![a-z])m(?=\d|\s|$)', 'เอ็ม ', text)
    text = re.sub(r'(?i)(?<![a-z])f(?=\d|\s|$)', 'เอฟ ', text)
    text = re.sub(r'\d+', normalize_number_token, text)

    return normalize_text(text)


assert prepare_target_text('M38 F42 Topic 69 Menu Ok') == 'เอ็ม สามสิบแปด เอฟ สี่สิบสอง หัวข้อ หกสิบเก้า เมนู โอเค'


processor = WhisperProcessor.from_pretrained(MODEL_NAME, language='thai', task='transcribe')
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=MODEL_DTYPE,
).to(DEVICE)
model.config.use_cache = False
model.generation_config.forced_decoder_ids = None


def prepare_examples(samples: list[dict[str, object]]) -> list[dict[str, object]]:
    prepared = []

    for sample in samples:
        input_features = processor.feature_extractor(
            sample['audio'],
            sampling_rate=sample['sampling_rate'],
            return_tensors='pt',
        ).input_features[0]
        target_text = prepare_target_text(sample['sentence'])

        prepared.append(
            {
                'input_features': input_features,
                'sentence': sample['sentence'],
                'target_text': target_text,
            }
        )

    return prepared


dev_items = prepare_examples(dev_samples)

print(f'Prepared {len(dev_items)} validation items')


Loading weights: 100%|██████████| 479/479 [00:01<00:00, 465.64it/s]


Prepared 20 validation items


## Inspect Results

These are baseline results only. No fine-tuning code exists in this notebook.


In [11]:
def evaluate_model(items: list[dict[str, object]], label: str) -> pd.DataFrame:
    model.eval()
    rows = []

    for index, item in enumerate(items, start=1):
        input_features = item['input_features'].unsqueeze(0).to(DEVICE, dtype=MODEL_DTYPE)

        with torch.no_grad():
            predicted_ids = model.generate(input_features, **GENERATION_KWARGS)

        prediction = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        reference = item['sentence']
        norm_reference = item['target_text']
        norm_prediction = prepare_target_text(prediction)

        rows.append(
            {
                'run': label,
                'sample': index,
                'reference': reference,
                'prediction': prediction,
                'target_reference': norm_reference,
                'cer': character_error_rate(norm_reference, norm_prediction),
                'wer': word_error_rate(norm_reference, norm_prediction),
            }
        )

    return pd.DataFrame(rows)


baseline_results = evaluate_model(dev_items, 'baseline')
summary = baseline_results.groupby('run')[['cer', 'wer']].mean().reset_index()

print('Best run by CER: baseline')
display(summary)
display(baseline_results[['run', 'sample', 'cer', 'wer', 'reference', 'prediction']])


[transformers] Both `max_new_tokens` (=80) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Best run by CER: baseline


,run,cer,wer
0,baseline,0.292503,1.421667


,run,sample,cer,wer,reference,prediction
0,baseline,1,0.171429,1.000000,ครับสวัสดีครับผม M สามสิบแปดครับ,สวัสดีครับผมM38ครัพ
1,baseline,2,0.200000,0.666667,สวัสดีครับ M สามสิบเก้าครับ,สวัสดีครับ เอม39
2,baseline,3,0.076923,0.666667,สวัสดีค่ะ F สี่สิบสองค่ะ,สวัสดีค่ะ F42 ค่า
3,baseline,4,0.217391,1.500000,Topic หกสิบเก้าชอบกินข้าวหรือชอบกินก๋วยเตี๋ยว,Topic 69 ชอบกินข้าว หรือชอมกินน์กืนเกียว
4,baseline,5,0.679245,1.000000,เป็นเรื่องใหญ่เหมือนกันนะส่วนส่วนผมชอบชอบข้าวค...,ส่วนผมชอบข้าวครับ
5,baseline,6,0.000000,0.000000,ทำไมถึงชอบข้าวคะ,ทำไมถึงชอบข้าวคะ
6,baseline,7,0.318841,2.000000,รู้สึกว่ากินข้าวมันอิ่มกว่าอ๋ออิ่มกว่าเหมือนกั...,รู้สึกว่ากินข้าวมันอิ่มกวะเหมือนกันครับ ชอบข้า...
7,baseline,8,0.517241,1.600000,นี่ชอบก๋วยเตี๋ยวค่ะเอาแล้วสองหนึ่งรอบนี้สองหนึ...,นี่ชอบวิตเดียวค่ะ?เอาแล้ว สอง หนึ่ง รอบนี้ ข้า...
8,baseline,9,0.342105,2.000000,แล้วชอบก๋วยเตี๋ยวนี่กินกินสมมุติว่าวันนึงอะเรา...,แล้วชอบกุษย์เตียวนี่กินกินน่ะสำมัววันหนึ่ง เรา...
9,baseline,10,0.090909,3.000000,โอ้สามมื้อก็ไม่ไหวเหมือนกันแต่ว่า,โอ้ สามมือก็ไม่ไหวเหมือนกัน แต่ว่า
